# Analisis Model 1D MT untuk Menemukan Model Optimal

Notebook ini melakukan analisis dua tahap:
1.  Membaca file log (`.logfile`) untuk menemukan iterasi inversi yang menghasilkan model paling optimal...
2.  Setelah iterasi terbaik ditemukan, notebook akan membaca file data resistivitas (`.iter`) yang sesuai...
3.  HARUS DILETAKKAN PADA DIRECTORY BERISI HASIL INVERSI PyGMI, DIRECTORY PER TITIK INVERSI 1D
4.  Dari sini anda dapat menggunakan outputnya untuk menampilkan pseudomap horizontal dan vertikal (lihat skrip lain)
5.  Pastikan model, iter, resp dan logfile ada semua

# 1. Import Packages & Initialization

In [9]:
import os
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

base_dir = r"C:\Users\Lenovo X390\Downloads\EDI\EDI\SSQ" 
input_dir = os.path.join(base_dir, "DATA")
output_dir = os.path.join(base_dir, "02. PLOT L-CURVE & 1D MODEL")
os.makedirs(output_dir, exist_ok=True)

# 2. Core Parser Functions

In [10]:
def parse_te_logfile(filepath):
    """Membaca logfile dari PyGMI/Occam untuk mendapatkan nilai Iteration, RMS, dan Roughness"""
    if not os.path.exists(filepath): return [], [], []
    with open(filepath, 'r', errors='ignore') as f: text = f.read()
    iters = [int(m) for m in re.findall(r'\*\*\s*Iteration\s+(\d+)\s*\*\*', text, re.IGNORECASE)]
    rms = [float(m) for m in re.findall(r'and is\s*=\s*([0-9.eE+-]+)', text, re.IGNORECASE)]
    rough = [float(m) for m in re.findall(r'Roughness\s*is\s*=\s*([0-9.eE+-]+)', text, re.IGNORECASE)]
    n = min(len(iters), len(rms), len(rough))
    return iters[:n], rms[:n], rough[:n]

def read_iter(filepath):
    """Membaca file .iter untuk mendapatkan array resistivitas"""
    if not os.path.exists(filepath): return None
    with open(filepath, 'r', errors='ignore') as f: text = f.read()
    pc_match = re.search(r'PARAM COUNT:\s*(\d+)', text, re.IGNORECASE)
    pc = int(pc_match.group(1)) if pc_match else None
    res_vals = [float(v) for line in text.splitlines() if not line.strip().startswith('!') and ':' not in line for v in line.split()]
    return 10 ** np.array(res_vals[:pc] if pc else res_vals)

def find_elbow_iteration(rms, rough):
    """Mencari iterasi 'Siku/L-Curve' (kompromi terbaik RMS vs Roughness)"""
    rms_arr = np.array(rms)
    rough_arr = np.array(rough)
    norm_rms = (rms_arr - rms_arr.min()) / (rms_arr.max() - rms_arr.min() + 1e-8)
    norm_rough = (rough_arr - rough_arr.min()) / (rough_arr.max() - rough_arr.min() + 1e-8)
    distances = np.sqrt(norm_rms**2 + norm_rough**2)
    return int(np.argmin(distances))

def read_model1d_cleaned(filepath):
    """Membaca kedalaman dari Model1D dan membuang Air Layer (kedalaman < 0 m)"""
    if not os.path.exists(filepath): return None
    with open(filepath, 'r', errors='ignore') as f: lines = f.readlines()
    depths = []
    started = False
    for line in lines:
        if '! top_depth' in line.lower():
            started = True
            continue
        if started and line.strip() and not line.startswith('!'):
            try:
                val = float(line.split()[0].replace(',', '.'))
                if val >= 0: # Hanya kedalaman permukaan ke bawah
                    depths.append(val)
            except ValueError:
                pass
    return np.array(depths)

# 3. Plotting Functions

In [11]:
def plot_lcurve(iters, rms, rough, best_i, best_r, best_ro, site, out_dir):
    """Plot Analisis L-Curve (RMS vs Roughness vs Iteration)"""
    plt.style.use('seaborn-v0_8-whitegrid')
    fig, ax = plt.subplots(1, 3, figsize=(18, 5))
    fig.suptitle(f"1D Occam Inversion L-Curve Analysis - Station: {site}", fontsize=16, fontweight='bold', color='#1f2b38', y=1.05)
    
    ax[0].plot(iters, rms, marker='o', color='#d62728', lw=2, ms=7)
    ax[0].axvline(best_i, color='gray', ls='--', alpha=0.7)
    ax[0].set(title="RMS Error vs Iteration", xlabel="Iteration Number", ylabel="RMS Error")
    
    ax[1].plot(iters, rough, marker='s', color='#1f77b4', lw=2, ms=7)
    ax[1].axvline(best_i, color='gray', ls='--', alpha=0.7)
    ax[1].set(title="Roughness vs Iteration", xlabel="Iteration Number", ylabel="Roughness")
    
    ax[2].plot(rough, rms, marker='^', color='#2ca02c', lw=2, ms=7)
    ax[2].scatter(best_ro, best_r, color='gold', s=200, zorder=5, ec='black', lw=1.5, label='Optimal Model (Elbow)')
    ax[2].set(title="RMS vs Roughness", xlabel="Roughness", ylabel="RMS Error")
    ax[2].legend(fontsize=11)
    
    plt.tight_layout()
    plt.savefig(os.path.join(out_dir, f"{site}_L-CURVE.png"), dpi=300, bbox_inches='tight')
    plt.close()

def plot_1d_cleaned(depths, res, site, iter_num, out_dir, max_depth=None):
    """Plotting 1D model dengan skala depth dan resistivitas otomatis sesuai data"""
    plt.style.use('seaborn-v0_8-whitegrid')
    fig, ax = plt.subplots(figsize=(6, 8))
    
    # Filter kedalaman jika max_depth ditentukan, jika tidak gunakan seluruh data
    if max_depth is not None:
        mask = depths <= max_depth
        plot_depths = depths[mask]
        plot_res = res[:len(plot_depths)]
    else:
        plot_depths = depths
        plot_res = res
    
    ax.step(plot_res, plot_depths, where='post', color='#b22222', lw=2.5)
    ax.set_xscale('log')
    ax.set_xlabel('Resistivity ($\Omega \cdot m$)', fontsize=12, fontweight='bold')
    ax.set_ylabel('Depth (m)', fontsize=12, fontweight='bold')
    ax.set_title(f'1D Resistivity Model - Station : {site}\nIteration {iter_num:03d}', fontsize=13, fontweight='bold')
    
    # Sumbu X (Resistivitas) otomatis menyesuaikan min & max dari data
    # Sumbu Y (Depth) dibalik: 0 m di atas dan batas bawah sesuai kedalaman maksimum data
    ax.set_ylim(plot_depths.max(), 0)
    
    ax.grid(True, which="both", ls='--', alpha=0.6)
    plt.tight_layout()
    plt.savefig(os.path.join(out_dir, f"{site}_1D_MODEL.png"), dpi=300, bbox_inches='tight')
    plt.close()

# 4. Main Batch Execution Loop

In [12]:
summary = []
if os.path.exists(input_dir):
    for item in os.listdir(input_dir):
        site_folder = os.path.join(input_dir, item)
        if not os.path.isdir(site_folder): continue
        site = item.split('_')[0]
        
        iters, rms, rough = parse_te_logfile(os.path.join(site_folder, "TE.logfile"))
        if not iters: continue
        
        # Cari iterasi siku/elbow L-Curve
        best_idx = find_elbow_iteration(rms, rough)
        best_i, best_r, best_ro = iters[best_idx], rms[best_idx], rough[best_idx]
        
        try:
            # Plot L-Curve
            plot_lcurve(iters, rms, rough, best_i, best_r, best_ro, site, output_dir)
            summary.append({"Station": site, "Optimal_Iter": best_i, "Optimal_RMS": round(best_r, 4), "Roughness": round(best_ro, 2)})
            
            # Cari file .iter yang sesuai
            iter_file = next((os.path.join(site_folder, f) for f in os.listdir(site_folder) if f.lower().endswith('.iter') and re.search(r'\d+', f) and int(re.search(r'\d+', f).group()) == best_i), None)
            res_data = read_iter(iter_file) if iter_file else None
            
            ground_depths = read_model1d_cleaned(os.path.join(site_folder, "Model1D"))
            
            if res_data is not None and ground_depths is not None:
                n_points = min(len(ground_depths), len(res_data))
                f_depths = ground_depths[:n_points]
                f_res = res_data[:n_points]
                
                # Simpan CSV hasil
                pd.DataFrame({'Depth': f_depths, 'Resistivity': f_res, 'RMS Error': best_r}).to_csv(os.path.join(output_dir, f"{site}.csv"), index=False)
                
                # Plot model 1D bersih (tanpa membatasi max_depth di 2000 m)
                plot_1d_cleaned(f_depths, f_res, site, best_i, output_dir)
                
        except Exception as e: 
            print(f"{site} Error: {e}")

if summary: 
    print(pd.DataFrame(summary).to_string(index=False))

Station  Optimal_Iter  Optimal_RMS  Roughness
 MT1-TE             4       1.3454       1.25
MT10-TE             4       1.4839       0.26
 MT2-TE             5       2.3561       0.88
 MT3-TE             4       1.1626       0.38
 MT4-TE             4       2.4957       1.95
 MT5-TE             3       1.5241       0.49
 MT6-TE             4       1.1729       0.42
 MT7-TE             5       0.9512       0.11
 MT8-TE             4       1.4040       1.29
 MT9-TE             4       1.3788       0.42
